### Colab environment setup
Run this once at the start of every new session.

In [ ]:
# Mount Drive so your uploaded files persist between sessions
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies Colab doesn't have pre-installed
!pip install geopandas -q
!pip install xlsxwriter -q

In [ ]:
!git clone https://github.com/SianReilly/Subnational-Statistics-and-Analysis.git

In [ ]:
#Set working directory to the location of your clustering and nearest neighbours code folder
import os
os.chdir('/content/Subnational-Statistics-and-Analysis/clustering_and_nearest_neighbours')

In [ ]:
#Imports the functions from other scripts
#If you make any changes to the underlying scripts you will need to rerun the code from here for those changes to apply
import yaml
import os

import sys
sys.path.insert(1, "../")

from src.data_functions import *
from src.neighbour_functions import *
from src.dissemination_functions import *
from yaml.loader import SafeLoader

In [ ]:
#Read in the config file
clustering_refactor_folder_path = os.path.abspath(os.path.join(os.path.realpath('__file__'), '../..'))
config_path = f"config.yaml".replace("\\", "/")
with open(config_path, encoding="utf-8") as f:
    loaded_config = yaml.load(f, Loader=SafeLoader)

In [ ]:
#Load data and split into individual metrics
#This will include all metrics specified in the config
datasets = import_data(
    loaded_config=loaded_config,
    cols_to_select=["AREACD", "AREANM","Indicator", "Value"],
    table_name=loaded_config["subnational_indicators_table_name"],
)

In [ ]:
#Cleans the data, including UTLA to LTLA imputation
for key, value in datasets.items():
    value = clean_groups(loaded_config, value)

In [ ]:
#Convert metrics into pivot tables, your specified data is now stored as tables['custom_metrics']
tables = {}
for key, value in datasets.items():
    tables[key] = metrics_to_table(value)

In [ ]:
#Set the max rows displayed as 500 if spot checking in script
pd.set_option('display.max_rows', 10)

In [ ]:
#Inspect the data to ensure all columns are present
tables['custom_metrics'].head()

In [ ]:
#Fuction subsets the data for the specified geography type using the names and codes file
#Names and codes file referred to as Geog_mapper in config
#This can be adapted through the config to run on any geography type or subset of geography
geog_df = get_desired_geography(
    loaded_config= loaded_config,
    df= tables['custom_metrics'],
)
geog_df

In [ ]:
#Removes any NAs if created by previous function
geog_df = geog_df.dropna(subset='AREACD')
geog_df

In [ ]:
#This function takes the dataset and computes pearsons correlation between all metrics
correlation_matrix = get_correlation_matrix(df= geog_df)
correlation_matrix

In [ ]:
#This function outputs a data frame of the winzorisation thresholds for each metric
#Currently set to 1st and 99th percentile, this can be altered
thresholds = get_winsorization_thresholds(
    df=geog_df,
    lower_threshold = 0.01,
    upper_threshold = 0.99,
)
thresholds

In [ ]:
#This function Winsorizes the data
#Winsorization deals with extreme values by setting any value above/below a certain percentile to the value of that percentile
#Currently set to 1st and 99th percentile, this can be altered
df_win = winsorize(
    df = geog_df,
    lower_threshold = 0.01,
    upper_threshold = 0.99,
)

In [ ]:
#This function calculates the distance between all points in the dataset and returns a dataframe of nearest neighbours
#The distance metric and number of neighbours can be specified
#This uses Euclidean distance, there are other options in the base function
#which can be changed with distance_metric, eg manhattan distance would be 'cityblock'
#This can be done with winsorized or data or original dataset by using either df_win or geog_df
#The outputted column '0' is the area of interest, followed by its neighbours in the other columns
euclidean_neighbours, euclidean_distances = nearest_neighbours(
    loaded_config=loaded_config,
    df= df_win,
    number= 20,
    distance_metric = "euclid",
)
euclidean_neighbours

In [ ]:
#This function creates a table of the variance of each metric included
variance_table = variance_analysis(
    loaded_config=loaded_config,
    df=geog_df,
)

In [ ]:
#This function creates a histogram of pairwise distances
#This histogram is saved in the "output" folder as a jpeg named "Histogram of Pairwise Distances"
pairwise_distances = visualize_pairwise_distances(
    loaded_config=loaded_config,
    df=geog_df,
)

In [ ]:
#This function the plot used for the elbow method of selecting the optimal number of neighbours
#This plot is saved in the "output" folder as a jpeg named "Elbow Method for Optimal k (Nearest Neighbors)"
#Set max neighbours to the maximum you want, max possible is number of areas available
Optimal_neighbours = find_optimal_neighbors(
    loaded_config=loaded_config,
    df=geog_df,
    max_neighbours = 361
)

In [ ]:
#This function exports all relevant data to a single xlsx file in the outputs folder
#Including the visualisations can be specified by the boolean operator include_maps
#If not all data is required, use frames parameter to specify desired sheets.
#File name must be specified with the ".xlsx" file type not included or it won't work
export_to_xlsx(
    loaded_config=loaded_config,
    frames = {'data': geog_df, 'Win_data':df_win, 'Win_thresholds':thresholds,
              'correlation_matrix': correlation_matrix,
              'euclidean_neighbours':euclidean_neighbours,'euclidean_distances':euclidean_distances,
              'variance_table': variance_table
             },
    file_name = "children_services_clustering_output",
    include_maps = False,
)

### Readable nearest neighbours
Translates area codes in `euclidean_neighbours` to local authority names using `names_and_codes_repo_25.csv`, and writes the result back into the exported xlsx as an additional sheet (`euclidean_neighbours_readable`) so it's saved permanently alongside the rest of the outputs, not just shown once in the notebook.

In [ ]:
import pandas as pd

output_path = f"{loaded_config['outputs_file_path']}/children_services_clustering_output.xlsx"

# Load and translate area codes to names
neighbours = pd.read_excel(output_path, sheet_name='euclidean_neighbours')
lookup = pd.read_csv(f"{loaded_config['inputs_file_path']}/names_and_codes_repo_25.csv")
code_to_name = dict(zip(lookup['LTLA25CD'], lookup['LTLA25NM']))

readable = neighbours.iloc[:, 1:].applymap(lambda code: code_to_name.get(code, code))
readable.columns = ['Area'] + [f'Neighbour_{i}' for i in range(1, readable.shape[1])]

# Write it back into the SAME xlsx file, as a new sheet, without touching the existing sheets
with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    readable.to_excel(writer, sheet_name='euclidean_neighbours_readable', index=False)

print("Saved. Sheets now in the file:")
print(pd.ExcelFile(output_path).sheet_names)
readable.head(10)